# 13 EVIE V2 on the pages notebook 11 did not cover (Kaggle T4 x2)

Same model, code, render, mask and scoring as notebook 11, V2 pass only (one-colour squares skip ViT and LLM).
Notebook 11 encoded the 3190 pages of the adaptive-K pools. This one encodes the other 1901 corpus pages (every `k3_input` page not in the
`k11-pools` pools) plus 100 pool pages notebook 11 already did (seed 0), so the local merge can check the two runs give the same scores before joining them
into one 220 x 5091 V2 matrix (columns in `colvec_keys.json` order).

Input: attach the `k3_input` and `k11-pools` datasets (as notebook 11). Settings: GPU T4 x2, Internet ON, "Save & Run All (Commit)".
Quick test: `SMOKE = True` encodes 10 pages.
Output: `k13_out_v2.zip` (folder `k13_out`) with `check.json`, `scores_v2.npy` (220 x 2001, columns `pool_page_ids.json`),
`pages_v2.jsonl`, `meta_v2.json`.
Local merge: `ondemand/routing_axes/evie_skip_blank_squares_merge_scores.py <unzipped k13_out>/k13_out`.


## 1. Settings

In [ ]:
INPUT_NAME = "pages.json"
POOLS_NAME = "k11_pools_xgbK214.json"
INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
OUT = f"{WORK_DIR}/k13_out"
EVIE_REPO = "https://github.com/Tencent/EVIE.git"
EVIE_PIN = "70b8a2be14de7285970a05ac0f670f768abad966"
MODEL_ID = "tencent/EVIE-4.5B"
TRANSFORMERS = "transformers==5.13.1"
DTYPE = "float16"
ATTN = "sdpa"
DPI = 144
SCORE_DIM = 2048
MVT = 1024
CHECK_PAGES = 5
OVERLAP = 100
CHECK_MIN_COS = 0.999
TIMING_PAGES = 100
WARMUP = 3
SEED = 0
TIME_BUDGET_H = 11.0
PASSES = ["v2"]
SMOKE = False
EVIE_DIR = f"{WORK_DIR}/EVIE"

## 2. Install (EVIE requirements, pinned commit) and input

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import ctypes
import glob
import hashlib
import json
import random
import subprocess
import sys
import threading
import time
from collections import defaultdict

if not os.path.exists(EVIE_DIR):
    subprocess.run(["git", "clone", "-q", EVIE_REPO, EVIE_DIR], check=True)
subprocess.run(["git", "-C", EVIE_DIR, "checkout", "-q", EVIE_PIN], check=True)
EVIE_COMMIT = subprocess.run(["git", "-C", EVIE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("EVIE commit", EVIE_COMMIT)
subprocess.run(["pip", "install", "-q", "-r", f"{EVIE_DIR}/requirements.txt", TRANSFORMERS, "pymupdf"], check=True)
sys.path.insert(0, f"{EVIE_DIR}/colpali")
found = sorted(glob.glob(f"{INPUT_ROOT}/**/{INPUT_NAME}", recursive=True))
found_pools = sorted(glob.glob(f"{INPUT_ROOT}/**/{POOLS_NAME}", recursive=True))
if "v3" in PASSES and "v2" not in PASSES:
    raise SystemExit("V3 needs V2 in the same run")
if not found or not found_pools:
    raise SystemExit(f"attach datasets containing {INPUT_NAME} (k3_input) and {POOLS_NAME}")
root = os.path.dirname(found[0])
pages = json.load(open(found[0]))
queries = json.load(open(f"{root}/queries.json"))
done = sorted({p for v in json.load(open(found_pools[0])).values() for p in v})
pool_pages = sorted({p["page_id"] for p in pages} - set(done)) + sorted(random.Random(SEED).sample(done, OVERLAP))
if SMOKE:
    pool_pages = pool_pages[::len(pool_pages) // 10][:10]
where = {p["page_id"]: (p["pdf"], p["index"]) for p in pages}
qidx = {q["query_id"]: i for i, q in enumerate(queries)}
unknown = [p for p in pool_pages if p not in where]
if unknown:
    raise SystemExit(f"{len(unknown)} pages not in k3_input, e.g. {unknown[:3]}")
os.makedirs(OUT, exist_ok=True)
print(len(pages), "pages", len(queries), "queries;", len(pool_pages), "pages to encode,", OVERLAP, "of them already in K11")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

## 3. Model: one copy per GPU

In [ ]:
import fitz
import numpy as np
import psutil
import torch
import transformers
from PIL import Image
from colpali_engine.models import ColQwen3_5, ColQwen3_5Processor
from transformers.vision_utils import get_vision_bilinear_indices_and_weights, get_vision_position_ids

print("transformers", transformers.__version__, "torch", torch.__version__)
proc = ColQwen3_5Processor.from_pretrained(MODEL_ID, max_num_visual_tokens=MVT)
models = globals().get("models") or []
for gpu in range(len(models), torch.cuda.device_count()):
    model = ColQwen3_5.from_pretrained(MODEL_ID, torch_dtype=getattr(torch, DTYPE), device_map=f"cuda:{gpu}",
                                       attn_implementation=ATTN).eval()
    model.enable_bidirectional_attention()
    models.append(model)
print(len(models), "model copies, max_pixels", proc.image_processor.max_pixels, "mrl_prefix", models[0].mrl_prefix)

## 4. Encoding: stock (as 09/10), hand-built with a keep mask over squares, masks, MaxSim

In [ ]:
def render(pid):
    pdf, index = where[pid]
    with fitz.open(os.path.join(root, "pdfs", pdf)) as doc:
        pix = doc.load_page(index).get_pixmap(dpi=DPI, colorspace=fitz.csRGB, alpha=False)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    fitz.TOOLS.store_shrink(100)
    return img


def encode(batch, model):
    batch = batch.to(model.device)
    model.rope_deltas = None
    with torch.no_grad():
        out = model(**batch)
    return out[SCORE_DIM][0][batch["attention_mask"][0].bool()]


def square_rows(batch):
    n = int(batch["image_grid_thw"][0].prod())
    return batch["pixel_values"][0][:n].reshape(n // 4, 4, 3, -1)


def blank_of(batch):
    px = square_rows(batch)
    return (px == px[:, :1, :, :1]).all(3).all(2).all(1).numpy()


def digest_of(batch):
    sq = square_rows(batch)
    rows = sq.reshape(sq.shape[0], -1).contiguous().numpy()
    return np.array([int.from_bytes(hashlib.blake2b(r.tobytes(), digest_size=8).digest(), "little") for r in rows],
                    dtype=np.uint64)


def grid_of(batch):
    return tuple(batch["image_grid_thw"][0].tolist())


def encode_kept(batch, model, keep, early):
    batch = batch.to(model.device)
    ids, am, grid = batch["input_ids"], batch["attention_mask"], batch["image_grid_thw"]
    vis = model.visual
    pv = batch["pixel_values"][0][:int(grid.prod())].type(vis.dtype)
    img = ids[0] == model.config.image_token_id
    keep = torch.as_tensor(keep, device=ids.device)
    with torch.no_grad():
        emb = model.get_input_embeddings()(ids)
        if early:
            rows = keep.repeat_interleave(vis.spatial_merge_unit)
            n = int(rows.sum())
            if n:
                idx, w = get_vision_bilinear_indices_and_weights(grid, num_grid_per_side=vis.num_grid_per_side,
                                                                 spatial_merge_size=vis.spatial_merge_size)
                pos = get_vision_position_ids(grid, vis.spatial_merge_size)
                feats = vis(pv[rows], grid_thw=grid, cu_seqlens=torch.tensor([0, n], dtype=torch.int32, device=ids.device),
                            position_ids=pos[rows], bilinear_indices=idx[:, rows], bilinear_weights=w[:, rows],
                            return_dict=True).pooler_output
        else:
            feats = vis(pv, grid_thw=grid, return_dict=True).pooler_output[keep]
        at = img.nonzero()[:, 0][keep]
        if len(at):
            emb[0, at] = feats.to(emb.dtype)
        tok = ~img
        tok[at] = True
        pos3, _ = model.get_rope_index(ids, mm_token_type_ids=batch["mm_token_type_ids"], image_grid_thw=grid,
                                       attention_mask=am)
        hidden = model.language_model(input_ids=None, inputs_embeds=emb[:, tok], attention_mask=am[:, tok],
                                      position_ids=pos3[:, :, tok], use_cache=False).last_hidden_state
        out = model.custom_text_proj(hidden)[0, :, :SCORE_DIM]
    return out / out.norm(dim=-1, keepdim=True).clamp(min=1e-12)


q_emb = [encode(proc.process_queries([q["query"]]), models[0]) for q in queries]
q_len = [e.shape[0] for e in q_emb]
q_end = np.cumsum(q_len)
q_cat = {g: torch.cat(q_emb).to(f"cuda:{g}") for g in range(len(models))}
seg = torch.repeat_interleave(torch.arange(len(q_emb)), torch.tensor(q_len))
print("query tokens", sum(q_len), "dim", q_emb[0].shape[1])


def score(page_emb, gpu):
    sim = (q_cat[gpu] @ page_emb.to(q_cat[gpu].dtype).T).max(dim=1).values.float().cpu()
    return torch.zeros(len(q_emb)).index_add_(0, seg, sim).numpy()


def score_one(page_emb, gpu, qi):
    q = q_cat[gpu][q_end[qi] - q_len[qi]:q_end[qi]]
    return float((q @ page_emb.to(q.dtype).T).max(dim=1).values.float().sum())

## 5. Keep-all check: hand-built path vs stock EVIE on 5 pool pages (stops on failure)

In [ ]:
rows = []
for pid in random.Random(SEED).sample(pool_pages, CHECK_PAGES):
    batch = proc.process_images([render(pid)])
    blank = blank_of(batch)
    ref = encode(batch, models[0]).float()
    row = {"page_id": pid, "tokens": int(ref.shape[0]), "squares": len(blank), "blank": int(blank.sum())}
    for name, early in (("late", False), ("early", True)):
        got = encode_kept(batch, models[0], np.ones(len(blank), bool), early).float()
        row[f"tokens_{name}"] = int(got.shape[0])
        row[f"min_cos_{name}"] = float(torch.nn.functional.cosine_similarity(ref, got, dim=-1).min()) if got.shape == ref.shape else float("nan")
        row[f"max_score_diff_{name}"] = float(np.abs(score(got, 0) - score(ref, 0)).max())
    v1 = encode_kept(batch, models[0], ~blank, False)
    v2 = encode_kept(batch, models[0], ~blank, True)
    row["tokens_v1"], row["tokens_v2"] = int(v1.shape[0]), int(v2.shape[0])
    row["corr_v1_full"] = float(np.corrcoef(score(v1, 0), score(ref, 0))[0, 1])
    row["corr_v2_full"] = float(np.corrcoef(score(v2, 0), score(ref, 0))[0, 1])
    rows.append(row)
    print(row)
json.dump(rows, open(f"{OUT}/check.json", "w"), indent=1)
bad = [r["page_id"] for r in rows
       if not (r["tokens_late"] == r["tokens_early"] == r["tokens"]
               and min(r["min_cos_late"], r["min_cos_early"]) >= CHECK_MIN_COS
               and r["tokens_v1"] == r["tokens_v2"] == r["tokens"] - r["blank"])]
if bad:
    raise SystemExit(f"hand-built path does not reproduce stock EVIE on {bad}; stopping")
if max(r["squares"] for r in rows) > MVT:
    raise SystemExit(f"squares per page {[r['squares'] for r in rows]} exceed MVT {MVT}: environment differs from 09")
print("keep-all check passed")

## 6. Pass runner: one thread per GPU, shared time budget

In [ ]:
started = time.time()
info, scores, left = {}, {}, {}
col = {p: i for i, p in enumerate(pool_pages)}


def run_pass(name, units, job):
    todo, lock, done, errors = list(units), threading.Lock(), [0], []
    log = open(f"{OUT}/{name}.jsonl", "w")

    def work(gpu):
        while True:
            with lock:
                if not todo or time.time() - started > TIME_BUDGET_H * 3600:
                    return
                unit = todo.pop(0)
            t = time.time()
            try:
                out = job(unit, gpu)
            except Exception as e:
                with lock:
                    errors.append(f"{unit!r:.120}: {e!r}")
                    todo.clear()
                return
            with lock:
                for r in out:
                    log.write(json.dumps(r | {"gpu": gpu, "seconds": round(time.time() - t, 3)}) + "\n")
                done[0] += 1
                if done[0] % 500 == 0:
                    log.flush()
                    ctypes.CDLL("libc.so.6").malloc_trim(0)
                    print(f"{name}: {done[0]} of {len(units)}, {time.time() - started:.0f}s, "
                          f"RAM {psutil.virtual_memory().percent:.0f}%")

    threads = [threading.Thread(target=work, args=(g,)) for g in range(len(models))]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    log.close()
    if errors:
        raise SystemExit(f"{name} failed: {errors[0]}")
    left[name] = len(todo)
    print(f"{name}: {done[0]} of {len(units)} done, {time.time() - started:.0f}s"
          + (f"; time budget hit, {len(todo)} left" if todo else ""))


def describe(pid):
    batch = proc.process_images([render(pid)])
    return grid_of(batch), blank_of(batch), digest_of(batch)

## 7. V2 (one-colour, skip early) over the listed pages

In [ ]:
def v2_job(pid, gpu):
    batch = proc.process_images([render(pid)])
    blank = blank_of(batch)
    emb = encode_kept(batch, models[gpu], ~blank, True)
    scores["v2"][:, col[pid]] = score(emb, gpu)
    return [{"page_id": pid, "squares": len(blank), "kept": int((~blank).sum()), "tokens": int(emb.shape[0])}]


if "v2" in PASSES:
    scores["v2"] = np.full((len(queries), len(pool_pages)), np.nan, dtype=np.float32)
    run_pass("pages_v2", pool_pages, v2_job)
    np.save(f"{OUT}/scores_v2.npy", scores["v2"])
    json.dump(pool_pages, open(f"{OUT}/pool_page_ids.json", "w"))

## 11. Save and download

In [ ]:
RUN = "_".join(PASSES)
json.dump({"model": MODEL_ID, "evie_commit": EVIE_COMMIT, "transformers": transformers.__version__, "torch": torch.__version__,
           "dtype": DTYPE, "attn": ATTN, "dpi": DPI, "score_dim": SCORE_DIM, "mvt": MVT, "pages_file": POOLS_NAME,
           "n_pages": len(pool_pages), "smoke": SMOKE, "seed": SEED, "batch": 1, "left": left, "passes": PASSES,
           "hours": round((time.time() - started) / 3600, 2), "gpu": torch.cuda.get_device_name(0),
           "n_gpus": len(models)}, open(f"{OUT}/meta_{RUN}.json", "w"), indent=1)
from IPython.display import FileLink, display
subprocess.run(["zip", "-qr", f"{WORK_DIR}/k13_out_{RUN}.zip", os.path.basename(OUT)], cwd=WORK_DIR, check=True)
print(f"k13_out_{RUN}.zip", f"{os.path.getsize(f'{WORK_DIR}/k13_out_{RUN}.zip') / 2**20:.0f} MB")
display(FileLink(f"k13_out_{RUN}.zip"))